# M3 Demo: Hybrid Retrieval
Vector search + BM25 + RRF + Voyage Rerank

In [ ]:
import os
import time
import json
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ["VERIDIAN_BASE_URL"]
API_KEY  = os.environ["VERIDIAN_API_KEY"]
AGENT_ID = os.environ["VERIDIAN_AGENT_ID"]
QUERY    = os.environ.get("DEMO_QUERY", "What is the refund policy?")

print(f"Agent: {AGENT_ID}")
print(f"Query: {QUERY}")

In [ ]:
resp = requests.post(
    f"{BASE_URL}/agents/{AGENT_ID}/query",
    headers={"X-API-Key": API_KEY},
    json={"query": QUERY},
)
resp.raise_for_status()
job_id = resp.json()["job_id"]
events_url = resp.json()["events_url"]
print(f"Job dispatched: {job_id}")
print(f"Monitor: {BASE_URL}{events_url}")

In [ ]:
def poll_query_complete(base_url, job_id, api_key, timeout=60):
    deadline = time.time() + timeout
    while time.time() < deadline:
        resp = requests.get(
            f"{base_url}/jobs/{job_id}/events",
            headers={"X-API-Key": api_key},
            stream=True,
            timeout=30,
        )
        for line in resp.iter_lines():
            if line and line.startswith(b"data:"):
                event = json.loads(line[5:])
                if event.get("event_type") == "query.complete":
                    return event["payload"]
        time.sleep(1)
    raise TimeoutError(f"query.complete not received within {timeout}s")

result = poll_query_complete(BASE_URL, job_id, API_KEY, timeout=60)
print(f"Strategy used: {result['strategy_used']}")
print(f"Results count: {len(result['results'])}")

In [ ]:
vector_df = pd.DataFrame(result["trace"]["vector_candidates"])
if not vector_df.empty:
    display(vector_df[["chunk_id", "cosine_score", "content"]].head(10))
else:
    print("No vector candidates (ensure M2 data is ingested)")

In [ ]:
bm25_df = pd.DataFrame(result["trace"]["bm25_candidates"])
if not bm25_df.empty:
    display(bm25_df[["chunk_id", "bm25_score", "content"]].head(10))
else:
    print("No BM25 candidates (ensure M2 data has text matching query keywords)")

In [ ]:
fused_df = pd.DataFrame(result["trace"]["fused_candidates"])
if not fused_df.empty:
    display(fused_df[["chunk_id", "rrf_score", "cosine_score", "bm25_score",
                       "vector_rank", "bm25_rank", "content"]].head(10))
else:
    print("No fused candidates")

In [ ]:
final_df = pd.DataFrame(result["results"])
if not final_df.empty:
    final_df["rerank_delta"] = final_df["rerank_score"] - final_df["rrf_score"].fillna(0)
    display(final_df[["chunk_id", "rerank_score", "rrf_score", "rerank_delta", "content"]].head(10))
else:
    print("No final results")

In [ ]:
vector_ids = set(r["chunk_id"] for r in result["trace"]["vector_candidates"][:5])
bm25_ids   = set(r["chunk_id"] for r in result["trace"]["bm25_candidates"][:5])
overlap    = len(vector_ids & bm25_ids)
print(f"Top-5 overlap between vector and BM25: {overlap}/5")
print("Expected: overlap < 5 for a query where semantic != keyword match")
if bm25_ids and vector_ids:
    assert overlap < len(vector_ids), "Vector and BM25 top-5 are identical — expected divergence"
    print("Vector and BM25 produce meaningfully different candidate sets")
else:
    print("Warning: One or both candidate sets are empty — ensure M2 data is ingested first")